In [1]:
import pandas as pd              # Para análisis de datos
import numpy as np               # Para cálculos numéricos
import matplotlib.pyplot as plt  # Para gráficos
import seaborn as sns            # Para gráficos bonitos

Este programa emplea el método Símplex de Nelder-Mead para optimización de funciones $f(\vec{x})$, donde $\rm{dim}(\vec{x}) = N$. Este método emplea técnicas geométricas para hallar el mínimo de nuestra función. Los diferentes procedimientos a realizar son: reflexión, expansión, contracción, encogimiento...

Si la dimensión de $\vec{x}$ es $N$, entonces necesitamos para arrancar con el algoritmo $N+1$ puntos de dimensión $N$ cada uno de ellos. ChatGPT sugiere conocer un $\vec{x_0}$ y luego desplazarlo una distancia $h$ en cada coordenada. Por ejemplo, si tenemos $P_1 = (x_0,y_0)$, hacer $P_2 = (x_0+h,y_0)$ y $P_3 = (x_0,y_0+h)$. 

Luego hay que ordenar en función del $f(P_i)$ más pequeño. Si $f(P_i) < f(P_j) < f(P_k)$ (en 3D), entonces $P_{mejor} = P_i$, $P_{malo} = P_j$ y $P_{peor} = P_k$. 

Por último el algoritmo consiste en hacer:

(1) Reflexión $\rightarrow$ Si se mejora $\rightarrow$ (2) Expansión en la dirección de la reflexión


(3) Si la reflexión empeora las cosas $\rightarrow$ Acercar el punto peor al centroide


(4) Si ambos pasos empeoran la cosa $\rightarrow$ Reducción (acercar todos los puntos al $P_{mejor}$)


Matemáticamente y más detallado ver http://www.scholarpedia.org/article/Nelder-Mead_algorithm (procedimiento que voy a seguir). 

Función perturbación de $\vec{x_0}$:

In [2]:
def perturb_x0(x0,h):
    # x0 = punto de busqueda inicial
    # h = parametro de aumento inicial (x0+h para los distintos elementos de x0)

    N = len(x0)
    X = np.tile(x0, (N+1, 1)) # Pongo x0 en las N+1 filas de X

    for k in range (0,N,1): # Perturbacion de las componentes de x0. En cada fila k de X perturbo la componente k de x0.
        X[k][k] = x0[k] + h

    return X

Funcion de ordenacion de puntos en función del valor de $f(\vec{x})$:

In [3]:
def orden(func, P):
    # P = matriz (N+1, N)
    # func = función que recibe vector y devuelve escalar (funcion a optimizar)
    
    valfunc = np.array([func(p) for p in P])  # calculo todos los valores
    ind_ord = np.argsort(valfunc)  # indices que ordenarían de menor a mayor
    
    P_ord = P[ind_ord] 
    #P[0] tiene el f(x) mas pequeño (punto optimo) y P[N] es valor mas grande de f(x) (punto peor)
    
    return P_ord

Algoritmo de Nelder-Mead:

In [4]:
def NelderMead(func,x0,h,alpha,beta,gamma,delta,maxit,tol):
    # func = funcion que depende del vector x
    # x0 = punto de busqueda inicial
    # h = parametro de aumento inicial (x0+h para los distintos elementos de x0)
    # alpha, beta, gamma, delta = parametros geometricos de busqueda (normalmente Alpha=1,Beta=0.5,Gamma=2,Delta=0.5)
    # maxit = numero maximo de iteraciones
    # tol = tolerancia de los puntos. Si tol > std(f(x_i)) se para el algoritmo.

    N = len(x0)

    X0_perturb = perturb_x0(x0,h) # Perturbacion del punto inicial (matriz)
    
    P_ord = orden(func,X0_perturb) # Ordenacion inicial de los puntos iniciales (matriz). P[0] punto optimo y P[N] punto peor

    iter = 0
    sigmafx = np.std(np.array([func(p) for p in P_ord])) # desviacion estandar de los N+1 vectores de P

    while iter < maxit and sigmafx > tol:
        Sum = 0
        for k in range (0,N,1): # No llego hasta el N+1 ya que el vector P[N] (peor) no lo cuento para el centroide
            Sum = Sum + P_ord[k] 

        Centroid = Sum/N

        Reflexion = Centroid + alpha * (Centroid - P_ord[N]) #P[N] es el punto peor

        if func(P_ord[0]) <= func(Reflexion) and func(P_ord[N-1]) > func(Reflexion):
            P_ord[N] = Reflexion # Sustituyo el punto peor de antes por el punto reflejado
            P_ord = orden(func,P_ord) # Reordeno el vector P_ord al haber cambiado un punto
            iter = iter + 1
            sigmafx = np.std(np.array([func(p) for p in P_ord]))

        else:
            if func(P_ord[0]) > func(Reflexion):
                Expansion = Centroid + gamma * (Reflexion - Centroid)
                
                if func(Expansion) < func(Reflexion):
                    P_ord[N] = Expansion
                    P_ord = orden(func,P_ord)
                    iter = iter + 1
                    sigmafx = np.std(np.array([func(p) for p in P_ord]))
                else: 
                    P_ord[N] = Reflexion
                    P_ord = orden(func,P_ord)
                    iter = iter + 1
                    sigmafx = np.std(np.array([func(p) for p in P_ord]))
            
            else:
                if func(Reflexion) >= func(P_ord[N-1]):
                    if func(Reflexion) >= func(P_ord[N-1]) and func(Reflexion) < func(P_ord[N]):
                        ContracOut = Centroid + beta * (Reflexion - Centroid) 

                        if func(ContracOut) <= func(Reflexion):
                            P_ord[N] = ContracOut
                            P_ord = orden(func,P_ord)
                            iter = iter + 1
                            sigmafx = np.std(np.array([func(p) for p in P_ord]))
                        else:
                            for k in range (1,N+1,1): #shrinking
                                P_ord[k] = P_ord[0] + delta * (P_ord[k] - P_ord[0])

                            P_ord = orden(func,P_ord)
                            iter = iter + 1
                            sigmafx = np.std(np.array([func(p) for p in P_ord]))                            

                    else:
                        if func(Reflexion) >= func(P_ord[N]):
                            ContracIn = Centroid + beta * (P_ord[N] - Centroid)

                            if func(ContracIn) < func(P_ord[N]):
                                P_ord[N] = ContracIn
                                P_ord = orden(func,P_ord)
                                iter = iter + 1
                                sigmafx = np.std(np.array([func(p) for p in P_ord])) 
                            else:
                                for k in range (1,N+1,1): #shrinking
                                    P_ord[k] = P_ord[0] + delta * (P_ord[k] - P_ord[0])

                                P_ord = orden(func,P_ord)
                                iter = iter + 1
                                sigmafx = np.std(np.array([func(p) for p in P_ord]))

    return P_ord[0],func(P_ord[0]),iter, sigmafx



Vamos a poner a prueba nuestro algoritmo para la función $f(x,y,x) = (x-1)^2 + (y+2)^2 + (z-0.5)^2$. Analíticamente la función alcanza el mínimo en $f(1,-2,0.5)=0$.

In [6]:
def funcion(v):
    val = (v[0]-1)**2 + (v[1]+2)**2 + (v[2]-0.5)**2
    return val

# Parámetros:

x0 = np.array([0.0,0.0,0.0]) # Importante poner el punto decimal para que calcule como floats y no como ints
h = 1
a = 1
b = 0.5
g = 2
d = 0.5
maxit = 1000
tol = 10**(-7)

vmin,fmin,iter,sigmafx = NelderMead(funcion,x0,h,a,b,d,g,maxit,tol)

print('El minimo se encuentra en el punto vmin =',vmin, 'y f(vmin) =' ,fmin, '.')
print('El numero de iteraciones ha sido ', iter, ' y la desviacion estandar estimada ha sido ', sigmafx, '.')


El minimo se encuentra en el punto vmin = [ 0.99982819 -2.00019986  0.49993674] y f(vmin) = 7.346339530413964e-08 .
El numero de iteraciones ha sido  45  y la desviacion estandar estimada ha sido  7.391686554602339e-08 .


CRÉDITOS: http://www.scholarpedia.org/article/Nelder-Mead_algorithm